# Phase 07C: Export Parity trên Validation & CPU Reference Benchmark

Phase 07C kiểm định các graph TorchScript FP32 đã xuất ở Phase 07B trên một cohort
cố định gồm 100 ảnh **Karpathy Validation**, đồng thời thực hiện benchmark CPU tham
chiếu ở batch 1.

Nguyên tắc khóa:

- chỉ dùng checkpoint đã chọn: T1–S1 seed 123, T2–S2 seed 2026, T3–S3 seed 42;
- không huấn luyện, không chọn lại checkpoint, không lượng tử hóa;
- kiểm tra eager PyTorch so với TorchScript trên đúng preprocessing của từng Student;
- gate FP32: sai số tuyệt đối cực đại của encoder và decoder không vượt `1e-4`;
- nếu frozen trace của TinyViT không đạt trên ảnh thực, notebook chỉ được phép
  thử lại bằng trace không freeze; không đổi trọng số và không nới ngưỡng;
- đo CPU một luồng: 10 warm-up, 30 lượt đo/cặp; đây chỉ là desktop reference,
  không thay thế benchmark trên thiết bị Android vật lý;
- không đọc caption Validation; Held-out Test luôn `SEALED`.

## Kaggle inputs bắt buộc

1. Phase 07B Portable Model Factory Export.
2. Phase 02A Shared Student Pipeline.
3. Phase 01 MS COCO Data Contract.
4. Phase 04D T1–S1 Sequence-KD seed 123.
5. Phase 04L T2–S2 Sequence-KD seed 2026.
6. Phase 04Q T3–S3 Logit-KD seed 42.
7. Dataset `mscoco2014-1` chứa thư mục `val2014`.

## Runtime

- Accelerator: `None / CPU`.
- Internet: `Off`.
- Không gắn output Test và không đọc `mscoco_test.parquet`.


## Cell 1 — Authorization, upstream integrity và runtime contract


In [1]:
import csv, gc, hashlib, importlib.util, json, os, platform, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//phase7_export_parity_validation_cpu_benchmark",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
REPAIR = OUT / "export_repairs"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, REPAIR, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

CPU_THREADS = int(os.environ.get("KLCN_CPU_THREADS", "1"))
assert CPU_THREADS == 1, "Phase 07C đã khóa benchmark CPU một luồng"
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
    INTEROP_THREADS = 1
except RuntimeError:
    INTEROP_THREADS = torch.get_num_interop_threads()
torch.set_grad_enabled(False)
torch.manual_seed(42)
np.random.seed(42)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name, root=NOTEBOOK_ROOT):
    matches = sorted(Path(root).rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents
    return target

def verify_phase7b_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file()
    assert sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(reference["rows"]) == int(marker["manifest_rows"])
    for row in rows:
        target = resolve_inside(marker_path.parent, row["relative_path"])
        assert target.is_file(), target
        assert target.stat().st_size == int(row["size_bytes"])
        assert sha256(target) == row["sha256"]
    return rows

def verify_phase2_manifest(marker_path, marker):
    reference = marker["manifests"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(marker["artifact_count"])
    for row in rows:
        target = resolve_inside(marker_path.parent, row["relative_path"])
        assert target.is_file(), target
        assert target.stat().st_size == int(row["size_bytes"])
        assert sha256(target) == row["sha256"]
    return rows

P7B_PATH = exactly_one("PHASE7B_PORTABLE_MODEL_FACTORY_EXPORT_COMPLETE.json")
AUTH_PATH = exactly_one("PHASE7C_EXPORT_PARITY_VALIDATION_AUTHORIZED.json")
P2_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P7B = read_json(P7B_PATH)
AUTH = read_json(AUTH_PATH)
P2 = read_json(P2_PATH)
P1 = read_json(P1_PATH)

assert P7B["status"] == "complete"
assert P7B["decision"] == "portable_factory_and_fp32_export_passed_phase7c_authorized"
assert AUTH["status"] == "authorized"
assert AUTH["constraints"]["new_training"] is False
assert AUTH["constraints"]["checkpoint_reselection"] is False
assert AUTH["constraints"]["fp32_only"] is True
assert AUTH["constraints"]["quantization"] is False
assert int(AUTH["constraints"]["validation_calibration_images"]) == 100
assert AUTH["constraints"]["held_out_test_opening"] is False
assert AUTH["test_set_status"] == "sealed"
assert AUTH["test_set_used"] is False
assert P2["status"] == "ready"
assert P2["data_policy"]["test_set_status"] == "sealed"
assert P2["data_policy"]["test_set_used"] is False
assert P1["status"] == "ready"
assert P1["test_set_status"] == "sealed"
assert P1["test_used"] is False

phase7b_rows = verify_phase7b_manifest(P7B_PATH, P7B)
phase2_rows = verify_phase2_manifest(P2_PATH, P2)

entry = {
    "schema_version": "phase7c_cell01_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "phase7b_manifest": f"{len(phase7b_rows)} / {len(phase7b_rows)} verified",
    "phase2_manifest": f"{len(phase2_rows)} / {len(phase2_rows)} verified",
    "runtime": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": "cpu",
        "intraop_threads": torch.get_num_threads(),
        "interop_threads": INTEROP_THREADS,
        "internet_required": False,
    },
    "protocol": {
        "validation_images": 100,
        "warmup_runs": 10,
        "measured_runs": 30,
        "batch_size": 1,
        "precision": "FP32",
        "encoder_max_abs_error_gate": 1e-4,
        "decoder_max_abs_error_gate": 1e-4,
    },
    "new_training_started": False,
    "quantization_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
}
write_json(AUDIT / "phase7c_cell01_entry.json", entry)

print("PHASE 07C CELL 1 — ENTRY")
print("Status                  : PASSED")
print("Phase 07B manifest      :", len(phase7b_rows), "/", len(phase7b_rows), "VERIFIED")
print("Phase 02A manifest      :", len(phase2_rows), "/", len(phase2_rows), "VERIFIED")
print("CPU threads             :", torch.get_num_threads())
print("New training            : NOT STARTED")
print("Quantization            : NOT STARTED")
print("Held-out Test           : SEALED")
print("Progress                : 1 / 5 code cells")


PHASE 07C CELL 1 — ENTRY
Status                  : PASSED
Phase 07B manifest      : 15 / 15 VERIFIED
Phase 02A manifest      : 38 / 38 VERIFIED
CPU threads             : 1
New training            : NOT STARTED
Quantization            : NOT STARTED
Held-out Test           : SEALED
Progress                : 1 / 5 code cells


## Cell 2 — Khóa cohort 100 ảnh Karpathy Validation


In [2]:
STARTED = time.perf_counter()
import pandas as pd

P1_ROOT = P1_PATH.parent
VAL_PARQUET = P1_ROOT / "parquet/mscoco_validation.parquet"
assert VAL_PARQUET.is_file(), VAL_PARQUET

# Chỉ đọc ba cột membership; tuyệt đối không đọc caption/token_ids.
validation_membership = pd.read_parquet(
    VAL_PARQUET,
    columns=["image_id", "karpathy_split", "source_partition"],
)
assert set(validation_membership["karpathy_split"].astype(str)) == {"val"}
assert set(validation_membership["source_partition"].astype(str)) == {"val2014"}
all_validation_ids = np.array(
    sorted(validation_membership["image_id"].astype(int).unique()),
    dtype=np.int64,
)
assert len(all_validation_ids) == 5000

rng = np.random.default_rng(42)
cohort_ids = sorted(
    int(value)
    for value in rng.choice(all_validation_ids, size=100, replace=False)
)
assert len(cohort_ids) == len(set(cohort_ids)) == 100

def image_filename(image_id):
    return f"COCO_val2014_{int(image_id):012d}.jpg"

# Ưu tiên đường dẫn đã khóa và các layout Kaggle phổ biến. Một số phiên
# mount dataset thành .../val2014/val2014 nên phải kiểm tra cả thư mục con.
known_val_roots = [
    INPUT_ROOT / "datasets/nguynthnhdan/mscoco2014-1/val2014",
    INPUT_ROOT / "datasets/nguynthnhdan/mscoco2014-1/val2014/val2014",
    INPUT_ROOT / "mscoco2014-1/val2014",
    INPUT_ROOT / "mscoco2014-1/val2014/val2014",
]

# Quét nông chỉ qua các cấp thư mục dataset, không duyệt 160 nghìn ảnh.
shallow_candidates = set(known_val_roots)
for pattern in [
    "*/val2014",
    "*/*/val2014",
    "*/*/*/val2014",
    "*/*/*/*/val2014",
]:
    for candidate in INPUT_ROOT.glob(pattern):
        if candidate.is_dir():
            shallow_candidates.add(candidate)
            shallow_candidates.add(candidate / "val2014")

val_candidates = sorted({
    candidate.resolve()
    for candidate in shallow_candidates
    if candidate.is_dir()
})
valid_val_roots = [
    candidate for candidate in val_candidates
    if all(
        (candidate / image_filename(image_id)).is_file()
        for image_id in cohort_ids
    )
]

# Fallback đầy đủ chỉ khi mọi layout nông đều không chứa cohort.
if not valid_val_roots:
    dataset_root = INPUT_ROOT / "datasets"
    search_root = dataset_root if dataset_root.is_dir() else INPUT_ROOT
    recursive_candidates = sorted({
        candidate.resolve()
        for candidate in search_root.rglob("val2014")
        if candidate.is_dir()
    })
    recursive_candidates = sorted(set(recursive_candidates) | {
        (candidate / "val2014").resolve()
        for candidate in recursive_candidates
        if (candidate / "val2014").is_dir()
    })
    valid_val_roots = [
        candidate for candidate in recursive_candidates
        if all(
            (candidate / image_filename(image_id)).is_file()
            for image_id in cohort_ids
        )
    ]

assert len(valid_val_roots) == 1, (
    "Cần đúng một thư mục val2014 chứa đủ cohort 100 ảnh; "
    f"tìm thấy {len(valid_val_roots)}: {valid_val_roots}; "
    f"ứng viên nông: {val_candidates}"
)
VAL_IMAGE_ROOT = valid_val_roots[0]
print("Resolved val2014 root    :", VAL_IMAGE_ROOT)

cohort_rows = []
for order, image_id in enumerate(cohort_ids):
    path = VAL_IMAGE_ROOT / image_filename(image_id)
    cohort_rows.append({
        "order": order,
        "image_id": image_id,
        "source_partition": "val2014",
        "filename": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })

COHORT_CSV = CONFIG / "phase7c_validation_image_cohort.csv"
with COHORT_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(cohort_rows[0].keys()))
    writer.writeheader()
    writer.writerows(cohort_rows)
write_json(CONFIG / "phase7c_validation_image_cohort.json", {
    "schema_version": "phase7c_validation_image_cohort_v1",
    "created_at_utc": now(),
    "selection": "NumPy Generator(PCG64), seed 42, sample without replacement",
    "population": "5,000 unique Karpathy Validation image IDs",
    "sample_size": 100,
    "caption_columns_read": False,
    "rows": cohort_rows,
})
write_json(AUDIT / "phase7c_cell02_validation_cohort.json", {
    "schema_version": "phase7c_cell02_validation_cohort_v1",
    "created_at_utc": now(),
    "status": "passed",
    "validation_population_images": len(all_validation_ids),
    "cohort_images": len(cohort_rows),
    "cohort_csv_sha256": sha256(COHORT_CSV),
    "caption_text_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

del validation_membership
gc.collect()

print("PHASE 07C CELL 2 — VALIDATION COHORT")
print("Status                  : PASSED")
print("Karpathy Validation     : 5,000 images")
print("Deterministic cohort    : 100 / 100 VERIFIED")
print("Caption text            : NOT READ")
print("Held-out Test           : SEALED / NOT READ")
print("Progress                : 2 / 5 code cells")


Resolved val2014 root    : /kaggle/input/datasets/nguynthnhdan/mscoco2014-1/val2014/val2014
PHASE 07C CELL 2 — VALIDATION COHORT
Status                  : PASSED
Karpathy Validation     : 5,000 images
Deterministic cohort    : 100 / 100 VERIFIED
Caption text            : NOT READ
Held-out Test           : SEALED / NOT READ
Progress                : 2 / 5 code cells


## Cell 3 — Factory, contracts, checkpoint và export integrity


In [3]:
STARTED = time.perf_counter()
import importlib.util

from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

P7B_ROOT = P7B_PATH.parent
P2_ROOT = P2_PATH.parent
P7B_ENTRY = read_json(P7B_ROOT / "audits/phase7b_cell01_entry.json")
EXPORT_REGISTRY = read_json(P7B_ROOT / "results/phase7b_export_registry.json")
FACTORY_AUDIT = read_json(P7B_ROOT / "audits/phase7b_cell02_factory.json")
EXPORT_AUDIT = read_json(P7B_ROOT / "audits/phase7b_cell04_export.json")

assert P7B_ENTRY["status"] == "passed"
assert EXPORT_AUDIT["status"] == "passed"
assert EXPORT_AUDIT["torchscript_fallback_passed_3_of_3"] is True
assert EXPORT_AUDIT["registered_fp32_thresholds"] == {
    "encoder_max_abs_error": 1e-4,
    "decoder_logit_max_abs_error": 1e-4,
}

FACTORY_PATH = resolve_inside(
    P7B_ROOT, FACTORY_AUDIT["factory_file"]["relative_path"]
)
assert sha256(FACTORY_PATH) == FACTORY_AUDIT["factory_file"]["sha256"]
spec = importlib.util.spec_from_file_location(
    "phase7b_portable_model_factory", FACTORY_PATH
)
factory = importlib.util.module_from_spec(spec)
spec.loader.exec_module(factory)

contract_paths = {
    "decoder": P2_ROOT / "configs/decoder/shared_transformer_decoder_contract.json",
    "S1": P2_ROOT / "configs/encoders/s1_encoder_adapter_contract.json",
    "S2": P2_ROOT / "configs/encoders/s2_encoder_adapter_contract.json",
    "S3": P2_ROOT / "configs/encoders/s3_encoder_adapter_contract.json",
}
preprocessing_paths = {
    student_id: (
        P2_ROOT
        / f"configs/preprocessing/{student_id.lower()}_preprocessing_contract.json"
    )
    for student_id in ["S1", "S2", "S3"]
}
for path in [*contract_paths.values(), *preprocessing_paths.values()]:
    assert path.is_file(), path
contracts = {key: read_json(path) for key, path in contract_paths.items()}
preprocessing_contracts = {
    key: read_json(path) for key, path in preprocessing_paths.items()
}

PAIR_TO_STUDENT = {"T1-S1": "S1", "T2-S2": "S2", "T3-S3": "S3"}
checkpoint_rows = []
for row in P7B_ENTRY["checkpoints"]:
    path = exactly_one(row["checkpoint_file"])
    assert path.stat().st_size == int(row["checkpoint_size_bytes"])
    assert sha256(path) == row["checkpoint_sha256"]
    checkpoint_rows.append({**row, "resolved_path": str(path)})
assert [row["pair"] for row in checkpoint_rows] == ["T1-S1", "T2-S2", "T3-S3"]

export_by_pair = {row["pair"]: row for row in EXPORT_REGISTRY["rows"]}
assert set(export_by_pair) == set(PAIR_TO_STUDENT)
portable_rows = []
for pair in PAIR_TO_STUDENT:
    row = export_by_pair[pair]
    assert row["torchscript_status"] == "passed"
    artifacts = {item["format"]: item for item in row["artifacts"]}
    assert {"torchscript_encoder", "torchscript_decoder"} <= set(artifacts)
    resolved = {}
    for key in ["torchscript_encoder", "torchscript_decoder"]:
        item = artifacts[key]
        path = resolve_inside(P7B_ROOT, item["relative_path"])
        assert path.stat().st_size == int(item["size_bytes"])
        assert sha256(path) == item["sha256"]
        resolved[key] = str(path)
    portable_rows.append({
        "pair": pair,
        "student_id": PAIR_TO_STUDENT[pair],
        "encoder_path": resolved["torchscript_encoder"],
        "decoder_path": resolved["torchscript_decoder"],
        "encoder_sha256": artifacts["torchscript_encoder"]["sha256"],
        "decoder_sha256": artifacts["torchscript_decoder"]["sha256"],
        "package_size_bytes": (
            int(artifacts["torchscript_encoder"]["size_bytes"])
            + int(artifacts["torchscript_decoder"]["size_bytes"])
        ),
        "encoder_export_variant": "phase7b_frozen_trace",
    })

def build_transform(student_id):
    contract = preprocessing_contracts[student_id]
    validation = contract["shared_geometry"]["validation"]
    return transforms.Compose([
        transforms.Resize(
            int(validation["resize_shorter"]),
            interpolation=InterpolationMode.BICUBIC,
            antialias=True,
        ),
        transforms.CenterCrop(int(validation["center_crop"])),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=contract["normalization"]["mean"],
            std=contract["normalization"]["std"],
        ),
    ])

validation_transforms = {
    student_id: build_transform(student_id)
    for student_id in ["S1", "S2", "S3"]
}

write_json(RESULT / "phase7c_portable_input_registry.json", {
    "schema_version": "phase7c_portable_input_registry_v1",
    "created_at_utc": now(),
    "rows": portable_rows,
})
write_json(AUDIT / "phase7c_cell03_portable_integrity.json", {
    "schema_version": "phase7c_cell03_portable_integrity_v1",
    "created_at_utc": now(),
    "status": "passed",
    "checkpoint_hashes_verified": "3 / 3",
    "torchscript_encoder_hashes_verified": "3 / 3",
    "torchscript_decoder_hashes_verified": "3 / 3",
    "preprocessing_contracts_verified": "3 / 3",
    "pretrained_download": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07C CELL 3 — PORTABLE INTEGRITY")
print("Status                  : PASSED")
print("Checkpoint hashes       : 3 / 3 VERIFIED")
print("TorchScript pairs       : 3 / 3 VERIFIED")
print("Preprocessing contracts : 3 / 3 VERIFIED")
print("Pretrained download     : DISABLED")
print("Held-out Test           : SEALED")
print("Progress                : 3 / 5 code cells")


PHASE 07C CELL 3 — PORTABLE INTEGRITY
Status                  : PASSED
Checkpoint hashes       : 3 / 3 VERIFIED
TorchScript pairs       : 3 / 3 VERIFIED
Preprocessing contracts : 3 / 3 VERIFIED
Pretrained download     : DISABLED
Held-out Test           : SEALED
Progress                : 3 / 5 code cells


## Cell 4 — Real-image parity và CPU reference benchmark


In [4]:
STARTED = time.perf_counter()
import statistics

try:
    import psutil
    PROCESS = psutil.Process()
except Exception:
    PROCESS = None

ENCODER_ATOL = float(
    EXPORT_AUDIT["registered_fp32_thresholds"]["encoder_max_abs_error"]
)
DECODER_ATOL = float(
    EXPORT_AUDIT["registered_fp32_thresholds"]["decoder_logit_max_abs_error"]
)
WARMUP_RUNS = 10
MEASURED_RUNS = 30
VOCABULARY_SIZE = 10080

def rss_mib():
    if PROCESS is None:
        return None
    return PROCESS.memory_info().rss / 2**20

def max_abs_error(left, right):
    return float(torch.max(torch.abs(left.float() - right.float())).item())

def percentile(values, percentile_value):
    return float(np.percentile(np.asarray(values, dtype=np.float64), percentile_value))

def make_token_probe(image_id):
    token_ids = torch.zeros((1, 31), dtype=torch.long)
    token_ids[0, 0] = 1
    for position, prime in enumerate([17, 31, 43, 59, 71, 89, 101], start=1):
        token_ids[0, position] = (
            (int(image_id) * prime + position * 97) % (VOCABULARY_SIZE - 4)
        ) + 4
    token_mask = torch.zeros((1, 31), dtype=torch.bool)
    token_mask[:, :8] = True
    return token_ids, token_mask

def load_verified_checkpoint(row):
    path = Path(row["resolved_path"])
    assert sha256(path) == row["checkpoint_sha256"]
    try:
        return torch.load(path, map_location="cpu", weights_only=True), "weights_only"
    except Exception as error:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        return payload, f"trusted_pickle_after_sha256:{type(error).__name__}"


checkpoint_by_pair = {row["pair"]: row for row in checkpoint_rows}
portable_by_pair = {row["pair"]: row for row in portable_rows}

# Phase 07B dùng trace + freeze. TinyViT có thể tích lũy sai số số học lớn hơn
# 1e-4 trên ảnh thực dù synthetic probe đã đạt. Không nới gate: nếu artifact
# frozen của T3 vượt gate, tạo trace không freeze và kiểm tra lại đủ 100 ảnh.
t3_checkpoint = checkpoint_by_pair["T3-S3"]
t3_portable = portable_by_pair["T3-S3"]
t3_transform = validation_transforms["S3"]
t3_eager_encoder = factory.build_encoder(
    "S3", contracts["S3"]
).cpu().eval()
t3_payload, t3_load_mode = load_verified_checkpoint(t3_checkpoint)
t3_load_result = t3_eager_encoder.load_state_dict(
    t3_payload["student_encoder_state"], strict=True
)
assert not t3_load_result.missing_keys and not t3_load_result.unexpected_keys
del t3_payload
gc.collect()

t3_original_encoder = torch.jit.load(
    t3_portable["encoder_path"], map_location="cpu"
).eval()
t3_original_errors = []
trace_image = None
with torch.inference_mode():
    for cohort in cohort_rows:
        image_path = VAL_IMAGE_ROOT / cohort["filename"]
        with Image.open(image_path) as image_handle:
            image = t3_transform(image_handle.convert("RGB")).unsqueeze(0)
        if trace_image is None:
            trace_image = image.clone()
        eager_memory = t3_eager_encoder(image)
        original_memory = t3_original_encoder(image)
        t3_original_errors.append(max_abs_error(eager_memory, original_memory))
        del image, eager_memory, original_memory

t3_original_max = max(t3_original_errors)
t3_repair_triggered = t3_original_max > ENCODER_ATOL
t3_repaired_max = None
t3_repair_path = None

if t3_repair_triggered:
    repair_dir = REPAIR / "t3_s3"
    repair_dir.mkdir(parents=True, exist_ok=True)
    t3_repair_path = repair_dir / "encoder_fp32_unfrozen_trace.ts"
    with torch.inference_mode():
        t3_repaired_encoder = torch.jit.trace(
            t3_eager_encoder,
            (trace_image,),
            strict=True,
            check_trace=True,
        ).eval()
    # Cố ý không gọi torch.jit.freeze(): đây là thay đổi kỹ thuật duy nhất.
    torch.jit.save(t3_repaired_encoder, str(t3_repair_path))
    del t3_repaired_encoder
    gc.collect()
    t3_repaired_encoder = torch.jit.load(
        str(t3_repair_path), map_location="cpu"
    ).eval()
    repaired_errors = []
    with torch.inference_mode():
        for cohort in cohort_rows:
            image_path = VAL_IMAGE_ROOT / cohort["filename"]
            with Image.open(image_path) as image_handle:
                image = t3_transform(
                    image_handle.convert("RGB")
                ).unsqueeze(0)
            eager_memory = t3_eager_encoder(image)
            repaired_memory = t3_repaired_encoder(image)
            repaired_errors.append(
                max_abs_error(eager_memory, repaired_memory)
            )
            del image, eager_memory, repaired_memory
    t3_repaired_max = max(repaired_errors)
    assert t3_repaired_max <= ENCODER_ATOL, {
        "pair": "T3-S3",
        "repair": "unfrozen_trace",
        "encoder_max_abs_error": t3_repaired_max,
        "registered_gate": ENCODER_ATOL,
    }
    t3_portable["encoder_path"] = str(t3_repair_path)
    t3_portable["encoder_sha256"] = sha256(t3_repair_path)
    t3_portable["encoder_export_variant"] = "phase7c_unfrozen_trace_repair"
    t3_portable["package_size_bytes"] = (
        t3_repair_path.stat().st_size
        + Path(t3_portable["decoder_path"]).stat().st_size
    )

T3_EXPORT_REPAIR = {
    "schema_version": "phase7c_t3_encoder_export_repair_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pair": "T3-S3",
    "reason": (
        "Phase 07B frozen trace exceeded the registered 1e-4 encoder "
        "max-absolute-error gate on the 100-image Validation cohort"
        if t3_repair_triggered
        else "Phase 07B frozen trace remained within the registered gate"
    ),
    "registered_encoder_max_abs_error_gate": ENCODER_ATOL,
    "original_frozen_trace_max_abs_error": t3_original_max,
    "repair_triggered": t3_repair_triggered,
    "repair_operation": (
        "torch.jit.trace without torch.jit.freeze"
        if t3_repair_triggered else None
    ),
    "repaired_trace_max_abs_error": t3_repaired_max,
    "repaired_artifact": (
        {
            "relative_path": t3_repair_path.relative_to(OUT).as_posix(),
            "size_bytes": t3_repair_path.stat().st_size,
            "sha256": sha256(t3_repair_path),
        }
        if t3_repair_path is not None else None
    ),
    "validation_images": 100,
    "checkpoint_changed": False,
    "weights_changed": False,
    "threshold_changed": False,
    "decoder_changed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(AUDIT / "phase7c_t3_encoder_export_repair.json", T3_EXPORT_REPAIR)
print(
    "T3-S3 export audit       : "
    f"frozen={t3_original_max:.3e} | "
    + (
        f"unfrozen={t3_repaired_max:.3e} | REPAIR PASSED"
        if t3_repair_triggered
        else "original artifact PASSED"
    )
)

del t3_eager_encoder, t3_original_encoder, trace_image
if "t3_repaired_encoder" in globals():
    del t3_repaired_encoder
gc.collect()

per_image_rows = []
summary_rows = []

for pair in ["T1-S1", "T2-S2", "T3-S3"]:
    pair_started = time.perf_counter()
    student_id = PAIR_TO_STUDENT[pair]
    checkpoint = checkpoint_by_pair[pair]
    portable = portable_by_pair[pair]
    transform = validation_transforms[student_id]
    rss_before_load = rss_mib()

    load_started = time.perf_counter_ns()
    eager_encoder = factory.build_encoder(
        student_id, contracts[student_id]
    ).cpu().eval()
    eager_decoder = factory.build_decoder(contracts["decoder"]).cpu().eval()
    payload, load_mode = load_verified_checkpoint(checkpoint)
    encoder_result = eager_encoder.load_state_dict(
        payload["student_encoder_state"], strict=True
    )
    decoder_result = eager_decoder.load_state_dict(
        payload["student_decoder_state"], strict=True
    )
    assert not encoder_result.missing_keys and not encoder_result.unexpected_keys
    assert not decoder_result.missing_keys and not decoder_result.unexpected_keys
    del payload
    gc.collect()
    ts_encoder = torch.jit.load(portable["encoder_path"], map_location="cpu").eval()
    ts_decoder = torch.jit.load(portable["decoder_path"], map_location="cpu").eval()
    model_load_ms = (time.perf_counter_ns() - load_started) / 1e6
    rss_after_load = rss_mib()
    peak_observed_rss = rss_after_load

    warmup_path = VAL_IMAGE_ROOT / cohort_rows[0]["filename"]
    with Image.open(warmup_path) as image_handle:
        warmup_image = transform(image_handle.convert("RGB")).unsqueeze(0)
    warmup_ids, warmup_mask = make_token_probe(cohort_rows[0]["image_id"])
    with torch.inference_mode():
        for _ in range(WARMUP_RUNS):
            warmup_memory = ts_encoder(warmup_image)
            _ = ts_decoder(warmup_ids, warmup_memory, warmup_mask)
    del warmup_image, warmup_ids, warmup_mask, warmup_memory

    encoder_errors = []
    decoder_errors = []
    top1_agreements = []
    preprocess_latencies = []
    encoder_latencies = []
    decoder_latencies = []
    end_to_end_latencies = []

    for sample_index, cohort in enumerate(cohort_rows):
        image_path = VAL_IMAGE_ROOT / cohort["filename"]
        pre_started = time.perf_counter_ns()
        with Image.open(image_path) as image_handle:
            image = transform(image_handle.convert("RGB")).unsqueeze(0)
        preprocess_ms = (time.perf_counter_ns() - pre_started) / 1e6
        token_ids, token_mask = make_token_probe(cohort["image_id"])

        with torch.inference_mode():
            eager_memory = eager_encoder(image)
            eager_logits = eager_decoder(token_ids, eager_memory, token_mask)

            encoder_started = time.perf_counter_ns()
            ts_memory = ts_encoder(image)
            encoder_ms = (time.perf_counter_ns() - encoder_started) / 1e6
            decoder_started = time.perf_counter_ns()
            ts_logits = ts_decoder(token_ids, ts_memory, token_mask)
            decoder_ms = (time.perf_counter_ns() - decoder_started) / 1e6

        assert tuple(eager_memory.shape) == tuple(ts_memory.shape) == (1, 49, 256)
        assert tuple(eager_logits.shape) == tuple(ts_logits.shape) == (1, 31, 10080)
        assert torch.isfinite(eager_memory).all() and torch.isfinite(ts_memory).all()
        assert torch.isfinite(eager_logits).all() and torch.isfinite(ts_logits).all()

        encoder_error = max_abs_error(eager_memory, ts_memory)
        decoder_error = max_abs_error(eager_logits, ts_logits)
        top1_agreement = float(
            (
                eager_logits.argmax(dim=-1)
                == ts_logits.argmax(dim=-1)
            ).float().mean().item()
        )
        measured = sample_index < MEASURED_RUNS
        end_to_end_ms = preprocess_ms + encoder_ms + decoder_ms

        encoder_errors.append(encoder_error)
        decoder_errors.append(decoder_error)
        top1_agreements.append(top1_agreement)
        if measured:
            preprocess_latencies.append(preprocess_ms)
            encoder_latencies.append(encoder_ms)
            decoder_latencies.append(decoder_ms)
            end_to_end_latencies.append(end_to_end_ms)

        current_rss = rss_mib()
        if current_rss is not None:
            peak_observed_rss = max(peak_observed_rss or current_rss, current_rss)
        per_image_rows.append({
            "pair": pair,
            "student_id": student_id,
            "image_id": cohort["image_id"],
            "sample_index": sample_index,
            "measured_for_latency": measured,
            "encoder_max_abs_error": encoder_error,
            "decoder_max_abs_error": decoder_error,
            "decoder_top1_agreement": top1_agreement,
            "preprocess_ms": preprocess_ms if measured else "",
            "encoder_ms": encoder_ms if measured else "",
            "decoder_ms": decoder_ms if measured else "",
            "end_to_end_ms": end_to_end_ms if measured else "",
        })

        del (
            image, token_ids, token_mask, eager_memory, eager_logits,
            ts_memory, ts_logits,
        )

    encoder_max = max(encoder_errors)
    decoder_max = max(decoder_errors)
    minimum_top1_agreement = min(top1_agreements)
    parity_passed = (
        encoder_max <= ENCODER_ATOL
        and decoder_max <= DECODER_ATOL
        and minimum_top1_agreement == 1.0
    )
    assert parity_passed, {
        "pair": pair,
        "encoder_max_abs_error": encoder_max,
        "decoder_max_abs_error": decoder_max,
        "minimum_top1_agreement": minimum_top1_agreement,
    }

    summary_rows.append({
        "pair": pair,
        "student_id": student_id,
        "seed": int(checkpoint["seed"]),
        "validation_images": len(cohort_rows),
        "warmup_runs": WARMUP_RUNS,
        "measured_runs": MEASURED_RUNS,
        "cpu_threads": CPU_THREADS,
        "precision": "FP32",
        "checkpoint_load_mode": load_mode,
        "encoder_export_variant": portable["encoder_export_variant"],
        "encoder_max_abs_error": encoder_max,
        "decoder_max_abs_error": decoder_max,
        "decoder_top1_agreement_min": minimum_top1_agreement,
        "parity_status": "passed",
        "model_load_ms": model_load_ms,
        "preprocess_p50_ms": percentile(preprocess_latencies, 50),
        "encoder_p50_ms": percentile(encoder_latencies, 50),
        "decoder_p50_ms": percentile(decoder_latencies, 50),
        "end_to_end_p50_ms": percentile(end_to_end_latencies, 50),
        "end_to_end_p90_ms": percentile(end_to_end_latencies, 90),
        "end_to_end_p95_ms": percentile(end_to_end_latencies, 95),
        "throughput_images_per_second": (
            1000.0 / statistics.mean(end_to_end_latencies)
        ),
        "package_size_bytes": int(portable["package_size_bytes"]),
        "package_size_mib": int(portable["package_size_bytes"]) / 2**20,
        "rss_before_load_mib": rss_before_load,
        "rss_after_load_mib": rss_after_load,
        "peak_observed_rss_mib": peak_observed_rss,
        "pair_elapsed_seconds": time.perf_counter() - pair_started,
    })
    print(
        f"{pair}: parity PASSED | encoder {encoder_max:.3e} | "
        f"decoder {decoder_max:.3e} | p50 {summary_rows[-1]['end_to_end_p50_ms']:.2f} ms"
    )

    del eager_encoder, eager_decoder, ts_encoder, ts_decoder
    gc.collect()

PER_IMAGE_CSV = RESULT / "phase7c_per_image_parity_and_latency.csv"
with PER_IMAGE_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(per_image_rows[0].keys()))
    writer.writeheader()
    writer.writerows(per_image_rows)

SUMMARY_CSV = RESULT / "phase7c_cpu_reference_summary.csv"
with SUMMARY_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)
write_json(RESULT / "phase7c_cpu_reference_summary.json", {
    "schema_version": "phase7c_cpu_reference_summary_v1",
    "created_at_utc": now(),
    "benchmark_scope": (
        "desktop CPU one-thread reference; not accepted as final physical-mobile result"
    ),
    "rows": summary_rows,
})
write_json(AUDIT / "phase7c_cell04_parity_benchmark.json", {
    "schema_version": "phase7c_cell04_parity_benchmark_v1",
    "created_at_utc": now(),
    "status": "passed",
    "fp32_thresholds": {
        "encoder_max_abs_error": ENCODER_ATOL,
        "decoder_max_abs_error": DECODER_ATOL,
        "greedy_token_sequence_exact_required": True,
    },
    "parity_passed": "3 / 3",
    "validation_images_per_pair": 100,
    "warmup_runs_per_pair": WARMUP_RUNS,
    "measured_runs_per_pair": MEASURED_RUNS,
    "desktop_cpu_reference_only": True,
    "accepted_as_final_mobile_result": False,
    "new_training_started": False,
    "quantization_started": False,
    "caption_text_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07C CELL 4 — PARITY & CPU BENCHMARK")
print("Status                  : PASSED")
print("Real Validation images  : 100 per pair")
print("FP32 parity             : 3 / 3 PASSED")
print("CPU benchmark           : 10 warm-up + 30 measured per pair")
print("Interpretation          : DESKTOP REFERENCE ONLY")
print("Held-out Test           : SEALED")
print("Progress                : 4 / 5 code cells")


/usr/local/lib/python3.13/dist-packages/timm/models/tiny_vit.py:390: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if H == self.window_size and W == self.window_size:
/usr/local/lib/python3.13/dist-packages/timm/models/tiny_vit.py:397: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  padding = pad_b > 0 or pad_r > 0
/usr/local/lib/python3.13/dist-packages/timm/models/tiny_vit.py:398: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as

T3-S3 export audit       : frozen=1.009e-04 | unfrozen=0.000e+00 | REPAIR PASSED
T1-S1: parity PASSED | encoder 8.750e-05 | decoder 2.432e-05 | p50 31.07 ms
T2-S2: parity PASSED | encoder 2.170e-05 | decoder 2.861e-05 | p50 82.65 ms
T3-S3: parity PASSED | encoder 0.000e+00 | decoder 0.000e+00 | p50 79.94 ms
PHASE 07C CELL 4 — PARITY & CPU BENCHMARK
Status                  : PASSED
Real Validation images  : 100 per pair
FP32 parity             : 3 / 3 PASSED
CPU benchmark           : 10 warm-up + 30 measured per pair
Interpretation          : DESKTOP REFERENCE ONLY
Held-out Test           : SEALED
Progress                : 4 / 5 code cells


## Cell 5 — Đóng Phase 07C và cấp quyền Phase 07D


In [5]:
STARTED = time.perf_counter()
assert len(summary_rows) == 3
assert all(row["parity_status"] == "passed" for row in summary_rows)
assert all(row["encoder_max_abs_error"] <= 1e-4 for row in summary_rows)
assert all(row["decoder_max_abs_error"] <= 1e-4 for row in summary_rows)
assert all(row["decoder_top1_agreement_min"] == 1.0 for row in summary_rows)
assert all(row["end_to_end_p50_ms"] > 0 for row in summary_rows)
assert all(row["throughput_images_per_second"] > 0 for row in summary_rows)

authorization = {
    "schema_version": "phase7d_mobile_optimization_readiness_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "required_next": (
        "Phase 07D mobile-runtime and quantization compatibility preflight "
        "using only Validation; no Test and no checkpoint reselection"
    ),
    "eligible_pairs": [
        {
            "pair": row["pair"],
            "student_id": row["student_id"],
            "seed": row["seed"],
            "fp32_parity_status": row["parity_status"],
            "encoder_export_variant": row["encoder_export_variant"],
            "package_size_mib": row["package_size_mib"],
            "cpu_reference_end_to_end_p50_ms": row["end_to_end_p50_ms"],
        }
        for row in summary_rows
    ],
    "constraints": {
        "new_training": False,
        "checkpoint_reselection": False,
        "validation_only": True,
        "held_out_test_opening": False,
        "quantization_requires_separate_fp32_quality_and_parity_gate": True,
        "physical_android_benchmark_required_for_final_claim": True,
        "desktop_cpu_result_is_final_mobile_result": False,
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(
    OUT / "PHASE7D_MOBILE_OPTIMIZATION_READINESS_AUTHORIZED.json",
    authorization,
)

completion = {
    "schema_version": "phase7c_export_parity_validation_cpu_benchmark_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "decision": "fp32_real_validation_parity_passed_phase7d_authorized",
    "validation_cohort": {
        "images": 100,
        "selection_seed": 42,
        "caption_text_read": False,
    },
    "fp32_parity": "3 / 3 passed",
    "technical_export_repair": T3_EXPORT_REPAIR,
    "cpu_reference_benchmark": {
        "threads": 1,
        "warmup_runs_per_pair": 10,
        "measured_runs_per_pair": 30,
        "accepted_as_final_mobile_result": False,
    },
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "quantization_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}

manifest_rows = []
for directory in [AUDIT, CONFIG, RESULT, REPAIR]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })
manifest_csv = MANIFEST / "phase7c_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / "phase7c_manifest.json"
write_json(manifest_json, {
    "schema_version": "phase7c_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
    encoding="utf-8",
)
completion["manifest"] = {
    "csv": {
        "relative_path": manifest_csv.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_csv),
        "rows": len(manifest_rows),
    },
    "json": {
        "relative_path": manifest_json.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_json),
    },
    "sha256sums": {
        "relative_path": sha_file.relative_to(OUT).as_posix(),
        "sha256": sha256(sha_file),
    },
}
completion["manifest_rows"] = len(manifest_rows)
completion["elapsed_seconds_cell05"] = time.perf_counter() - STARTED
write_json(
    OUT / "PHASE7C_EXPORT_PARITY_VALIDATION_CPU_BENCHMARK_COMPLETE.json",
    completion,
)

print("PHASE 07C — FINAL")
print("Status                  : COMPLETE")
print("Validation cohort       : 100 / 100")
print("FP32 parity             : 3 / 3 PASSED")
print("CPU reference           : COMPLETE")
print("Final mobile claim      : NOT YET — physical Android required")
print("New training            : NOT PERFORMED")
print("Quantization            : NOT PERFORMED")
print("Held-out Test           : SEALED / NOT USED")
print("Phase 07D               : AUTHORIZED")
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Progress                : 5 / 5 code cells")


PHASE 07C — FINAL
Status                  : COMPLETE
Validation cohort       : 100 / 100
FP32 parity             : 3 / 3 PASSED
CPU reference           : COMPLETE
Final mobile claim      : NOT YET — physical Android required
New training            : NOT PERFORMED
Quantization            : NOT PERFORMED
Held-out Test           : SEALED / NOT USED
Phase 07D               : AUTHORIZED
Manifest                : 12 / 12 VERIFIED
Progress                : 5 / 5 code cells
